# Automated Exam Evaluation System (NLP-based descriptive answer grading)
### This project aims to automate the grading process for student essays using Natural Language Processing (NLP) and Machine Learning. It utilises the ASAP-AES dataset to build a regressor that can predict essay scores based on the semantic content of the text.

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving asap-aes.zip to asap-aes.zip


In [ ]:
!unzip asap-aes.zip

Archive:  asap-aes.zip
  inflating: Essay_Set_Descriptions.zip  
  inflating: Training_Materials.zip  
  inflating: test_set.tsv            
  inflating: training_set_rel3.tsv   
  inflating: training_set_rel3.xls   
  inflating: training_set_rel3.xlsx  
  inflating: valid_sample_submission_1_column.csv  
  inflating: valid_sample_submission_1_column_no_header.csv  
  inflating: valid_sample_submission_2_column.csv  
  inflating: valid_sample_submission_5_column.csv  
  inflating: valid_set.tsv           
  inflating: valid_set.xls           
  inflating: valid_set.xlsx          


In [ ]:
!pip install sentence-transformers scikit-learn pandas

# Importing lib and model

In [ ]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

In [ ]:
!pip install sentence-transformers

###In order to process and understand the textual content of student essays, I use a pre-trained Sentence-BERT model. Sentence-BERT is a modification of the BERT transformer model that is specifically designed to generate semantic embeddings for sentences and paragraphs. These embeddings convert text into numerical vectors that capture the meaning and context of the essay.

### In this project, I used the model (all-MiniLM-L6-v2), which is a lightweight and efficient Sentence-BERT model that produces 384-dimensional vector representations of text.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

#Essay Set	Score Range
###Set 1	2 – 12
###Set 2	1 – 6
###Set 3	0 – 3
###Set 4	0 – 3
###Set 5	0 – 4
###Set 6	0 – 4
###Set 7	0 – 30
###Set 8	0 – 60

# For Set 1

In [ ]:
import pandas as pd

data = pd.read_csv(
    "training_set_rel3.tsv",
    sep="\t",
    encoding="latin-1"
)

data.head()

,essay_id,essay_set,essay,rater1_domain1,rater2_domain1,rater3_domain1,domain1_score,rater1_domain2,rater2_domain2,domain2_score,...,rater2_trait3,rater2_trait4,rater2_trait5,rater2_trait6,rater3_trait1,rater3_trait2,rater3_trait3,rater3_trait4,rater3_trait5,rater3_trait6
0,1,1,"Dear local newspaper, I think effects computer...",4,4,NaN,8,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1,"Dear @CAPS1 @CAPS2, I believe that using compu...",5,4,NaN,9,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,1,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",4,3,NaN,7,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,1,"Dear Local Newspaper, @CAPS1 I have found that...",5,5,NaN,10,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,1,"Dear @LOCATION1, I know having computers has a...",4,4,NaN,8,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
print(df.columns)

Index(['text', 'score'], dtype='object')


In [ ]:
df = data[data['essay_set'] == 1]
df = df[['essay', 'domain1_score']]

df = df.dropna()

df.head()

,essay,domain1_score
0,"Dear local newspaper, I think effects computer...",8
1,"Dear @CAPS1 @CAPS2, I believe that using compu...",9
2,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",7
3,"Dear Local Newspaper, @CAPS1 I have found that...",10
4,"Dear @LOCATION1, I know having computers has a...",8


In [ ]:
df.rename(columns={
    'essay': 'text',
    'domain1_score': 'score'
}, inplace=True)

In [ ]:
texts = df['text']
scores = df['score']

In [ ]:
embeddings = model.encode(
    texts.tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/56 [00:00<?, ?it/s]

### The dataset is divided into two parts: a training set and a testing set. In this project, I use an 80–20 split, where 80% of the data is used to train the model and 20% is used to test and evaluate the model's performance.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    embeddings,
    scores,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 1426
Testing samples: 357


###In this project, I used a Random Forest Regressor to predict the essay score based on the embeddings generated from the Sentence-BERT model. Random Forest is an ensemble machine learning algorithm that combines multiple decision trees to make more accurate predictions. Instead of relying on a single decision tree, Random Forest builds many trees during training and then averages their predictions to produce the final output.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

regressor = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

regressor.fit(X_train, y_train)

RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42)

###Mean Squared Error (MSE) is a common evaluation metric used for regression models. MSE measures the average of the squared differences between the actual scores given by human graders and the scores predicted by the model. A smaller MSE value indicates that the predicted scores are closer to the true scores, meaning the model performs better.

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

pred = regressor.predict(X_test)
mean_error_for_set1 = mean_squared_error(y_test, pred)

print("Mean Squared Error:", mean_squared_error(y_test, pred))
print("R2 Score:", r2_score(y_test, pred))

Mean Squared Error: 1.5694617647058822
R2 Score: 0.3105393201089205


# For set 2

In [ ]:
df = data[data['essay_set'] == 2]
df = df[['essay', 'domain1_score']]

df = df.dropna()

df.head()

,essay,domain1_score
1783,Certain materials being removed from libraries...,4
1784,Write a persuasive essay to a newspaper reflec...,1
1785,Do you think that libraries should remove cert...,2
1786,"In @DATE1's world, there are many things found...",4
1787,In life you have the 'offensive things'. The l...,4


In [ ]:
df.rename(columns={
    'essay': 'text',
    'domain1_score': 'score'
}, inplace=True)

In [ ]:
texts = df['text']
scores = df['score']

In [ ]:
embeddings = model.encode(
    texts.tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/57 [00:00<?, ?it/s]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    embeddings,
    scores,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 1440
Testing samples: 360


In [ ]:
from sklearn.ensemble import RandomForestRegressor

regressor = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

regressor.fit(X_train, y_train)

RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42)

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

pred = regressor.predict(X_test)
mean_error_for_set2 = mean_squared_error(y_test, pred)

print("Mean Squared Error:", mean_squared_error(y_test, pred))
print("R2 Score:", r2_score(y_test, pred))

Mean Squared Error: 0.42339541666666664
R2 Score: 0.2594465828114877


In [ ]:
data['essay_set'].unique()

array([1, 2, 3, 4, 5, 6, 7, 8])

In [ ]:
sample_index = 45
original_df_index = y_test.index[sample_index]

sample_essay = texts[original_df_index]
sample_actual_score = y_test.iloc[sample_index]
sample_predicted_score = pred[sample_index]

print(f"Original Essay:\n{sample_essay}\n")
print(f"Actual Score: {sample_actual_score}")
print(f"Predicted Score: {sample_predicted_score}")

Original Essay:
This is what's wrong with today's society. We are too soft.We're too worried about what other people might think if we say this or do that. I'm the kind of person that doesn't care about other peoples feelings. And I say that to an extent. If we're talking about my friends, family, my girlfriend, that's totally different. I show a lot of respect to those people on a daily basis. But as far as people I don't know, I could care less what their beliefs, opinions, and outlooks on life are. Maybe we should stop worrying about other people for once and just try to grow and prosper as a race.      Anybody ever think of that? If I had it my way, the @ORGANIZATION1 would not exist. @CAPS1, radio, books, movies, magazines would all be uncensored. Is life filtered and censored? Absolutely not! So why should those things be? I'm sick and tired of people constantly sheltering their children from lifes normalcies. They've gotta learn sometime. Life isn't lollipops and rainbows. It's 

# For Set 3

In [ ]:
df = data[data['essay_set'] == 3]
df = df[['essay', 'domain1_score']]

df = df.dropna()

df.head()

,essay,domain1_score
3583,The features of the setting affect the cyclist...,1
3584,The features of the setting affected the cycli...,2
3585,Everyone travels to unfamiliar places. Sometim...,1
3586,I believe the features of the cyclist affected...,1
3587,The setting effects the cyclist because of the...,2


In [ ]:
df.rename(columns={
    'essay': 'text',
    'domain1_score': 'score'
}, inplace=True)

In [ ]:
texts = df['text']
scores = df['score']

In [ ]:
embeddings = model.encode(
    texts.tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/54 [00:00<?, ?it/s]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    embeddings,
    scores,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 1380
Testing samples: 346


In [ ]:
from sklearn.ensemble import RandomForestRegressor

regressor = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

regressor.fit(X_train, y_train)

RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42)

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

pred = regressor.predict(X_test)
mean_error_for_set3 = mean_squared_error(y_test, pred)

print("Mean Squared Error:", mean_squared_error(y_test, pred))
print("R2 Score:", r2_score(y_test, pred))

Mean Squared Error: 0.349168780199363
R2 Score: 0.4224853250114402


In [ ]:
sample_index = 4
original_df_index = y_test.index[sample_index]

sample_essay = texts[original_df_index]
sample_actual_score = y_test.iloc[sample_index]
sample_predicted_score = pred[sample_index]

print(f"Original Essay:\n{sample_essay}\n")
print(f"Actual Score: {sample_actual_score}")
print(f"Predicted Score: {sample_predicted_score}")

Original Essay:
The features of the setting affects the cyclist in the story. In the story, it says, The water bottles contained only a few tantalizing sips. I could drop from a heatstroke. This statement clearly says that where the cyclist was riding, water was very hard to find and scare. He is talking about how he will die if he doesnt get water. Another example is that there are no buildings around and is deserted with no one around and many hills and roads. In the story it says, the terrain changing flat road was replaced by short rolling hills. This means that he was partially all by himself and he had no one to go to.

Actual Score: 2
Predicted Score: 2.12


In [ ]:
sample_index = 32
original_df_index = y_test.index[sample_index]

sample_essay = texts[original_df_index]
sample_actual_score = y_test.iloc[sample_index]
sample_predicted_score = pred[sample_index]

print(f"Original Essay:\n{sample_essay}\n")
print(f"Actual Score: {sample_actual_score}")
print(f"Predicted Score: {sample_predicted_score}")

Original Essay:
The setting clearly has a huge effect on the cyclist. The flat roads are easy and smooth to ride on, but as the road changes to short, rolling hills, the biking becomes more difficult. This is supported by the sentence that reads, "Over are long, crippling hill... This shows that the hill is a very challenging obstacle. He is also effected by the things that come with the change in terrain. For example, one sentence reads "...tumbleweeds crossed my path and a ridiculously large snake it really did look like a diamondbuck-blooked the majority of the pavement in front of me." This stresses how remote the area in which he is biking really is, and shows how it mentally and physically effects his trip.  

Actual Score: 2
Predicted Score: 2.13


# For Set 4

In [ ]:
df = data[data['essay_set'] == 4]
df = df[['essay', 'domain1_score']]

df = df.dropna()

df.head()

,essay,domain1_score
5309,The author concludes the story with this becau...,0
5310,The narrater has that in with Paragraph becuse...,0
5311,The author concludes the story with that passa...,3
5312,The author ended the story with this paragraph...,2
5313,The author concludes the story with this parag...,2


In [ ]:
df.rename(columns={
    'essay': 'text',
    'domain1_score': 'score'
}, inplace=True)

In [ ]:
texts = df['text']
scores = df['score']

In [ ]:
embeddings = model.encode(
    texts.tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/56 [00:00<?, ?it/s]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    embeddings,
    scores,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 1416
Testing samples: 354


In [ ]:
from sklearn.ensemble import RandomForestRegressor

regressor = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

regressor.fit(X_train, y_train)

RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42)

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

pred = regressor.predict(X_test)
mean_error_for_set4 = mean_squared_error(y_test, pred)

print("Mean Squared Error:", mean_squared_error(y_test, pred))
print("R2 Score:", r2_score(y_test, pred))

Mean Squared Error: 0.35490818455743883
R2 Score: 0.5912500431398138


In [ ]:
sample_index = 4
original_df_index = y_test.index[sample_index]

sample_essay = texts[original_df_index]
sample_actual_score = y_test.iloc[sample_index]
sample_predicted_score = pred[sample_index]

print(f"Original Essay:\n{sample_essay}\n")
print(f"Actual Score: {sample_actual_score}")
print(f"Predicted Score: {sample_predicted_score}")

Original Essay:
The author concludes the story with the paragraph because the geese and the hibiscus were both things that were familiar to Saeng They remind her of her homeland,When  the familiar things return ,she will have the strength to the test again and hopefully do better.The narrator says Overhead a of Canada geese flew by,their faint hunks clear and yes-fimiliar to Saeng now .This shows how the geese are a symbol of the fimilar things from her homeland that help Saeng feel more comfortable in her new home.

Actual Score: 2
Predicted Score: 1.97


In [ ]:
sample_index = 24
original_df_index = y_test.index[sample_index]

sample_essay = texts[original_df_index]
sample_actual_score = y_test.iloc[sample_index]
sample_predicted_score = pred[sample_index]

print(f"Original Essay:\n{sample_essay}\n")
print(f"Actual Score: {sample_actual_score}")
print(f"Predicted Score: {sample_predicted_score}")

Original Essay:
The authors choice of the last paragraph in the short story, Winter Hibiscus shows the hope for the future that seang feels. The winter and cold represent how she feels out of place and upset at not receiving her drivers license. In the spring she will have a piece of her homeland blooming and she will be warm again. She will feel ready to take her test again when they came back. This shows adapting to and accepting her new life away from her home and her memories. The last paragraph proves that she was strong enough to move past her hardships and move on

Actual Score: 2
Predicted Score: 2.165


# For Set 5

In [ ]:
df = data[data['essay_set'] == 5]
df = df[['essay', 'domain1_score']]

df = df.dropna()

df.head()

,essay,domain1_score
7079,"In this memoir of Narciso Rodriguez, @PERSON3'...",2
7080,Throughout the excerpt from Home the Blueprint...,2
7081,The mood the author created in the memoir is l...,3
7082,The mood created by the author is showing how ...,1
7083,The mood created in the memoir is happiness an...,3


In [ ]:
df.rename(columns={
    'essay': 'text',
    'domain1_score': 'score'
}, inplace=True)

In [ ]:
texts = df['text']
scores = df['score']

In [ ]:
embeddings = model.encode(
    texts.tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/57 [00:00<?, ?it/s]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    embeddings,
    scores,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 1444
Testing samples: 361


In [ ]:
from sklearn.ensemble import RandomForestRegressor

regressor = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

regressor.fit(X_train, y_train)

RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42)

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

pred = regressor.predict(X_test)
mean_error_for_set5 = mean_squared_error(y_test, pred)

print("Mean Squared Error:", mean_squared_error(y_test, pred))
print("R2 Score:", r2_score(y_test, pred))

Mean Squared Error: 0.3460765235457064
R2 Score: 0.6076805965118302


In [ ]:
sample_index = 16
original_df_index = y_test.index[sample_index]

sample_essay = texts[original_df_index]
sample_actual_score = y_test.iloc[sample_index]
sample_predicted_score = pred[sample_index]

print(f"Original Essay:\n{sample_essay}\n")
print(f"Actual Score: {sample_actual_score}")
print(f"Predicted Score: {sample_predicted_score}")

Original Essay:
I think the mood was happy and @CAPS1. Because They @CAPS2. The boy Loved his parents. They had to leave theirs friends. Some family there and They would miss them, but the parents wanted a better life for thier children. They were @CAPS1. to Move.

Actual Score: 1
Predicted Score: 1.095


In [ ]:
sample_index = 7
original_df_index = y_test.index[sample_index]

sample_essay = texts[original_df_index]
sample_actual_score = y_test.iloc[sample_index]
sample_predicted_score = pred[sample_index]

print(f"Original Essay:\n{sample_essay}\n")
print(f"Actual Score: {sample_actual_score}")
print(f"Predicted Score: {sample_predicted_score}")

Original Essay:
The mood in the author's story is happy and gratefull. Happy because even though it was a new country the came with a purpose and are living it with their culture which is past down family to family. "My mother and father came to this country with such couage, without any knowledge of the language or the culture. They came selflessly, as many immigrants do, to give their children a better life, even though it meant leaving behind their family, friends and careers in the country they loved." Also greatfull because he sees what his parents did and does for him so he could be happy. "I will always be gratefull to my parents for their love and sacrifice." "I was in the warmth of the kitchen in this humble house where a cuban feast always filled the air with not just scent and music but life and love!"

Actual Score: 3
Predicted Score: 2.785


# For Set 6

In [ ]:
df = data[data['essay_set'] == 6]
df = df[['essay', 'domain1_score']]

df = df.dropna()

df.head()

,essay,domain1_score
8884,There were many obstacles that the builders fa...,2
8885,"Him from the start, there would have been many...",3
8886,The builders of the Empire State Building face...,4
8887,In the passage The Mooring Mast by Marcia Amid...,1
8888,The builders of the Empire State Building face...,3


In [ ]:
df.rename(columns={
    'essay': 'text',
    'domain1_score': 'score'
}, inplace=True)

In [ ]:
texts = df['text']
scores = df['score']

In [ ]:
embeddings = model.encode(
    texts.tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/57 [00:00<?, ?it/s]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    embeddings,
    scores,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 1440
Testing samples: 360


In [ ]:
from sklearn.ensemble import RandomForestRegressor

regressor = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

regressor.fit(X_train, y_train)

RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42)

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

pred = regressor.predict(X_test)
mean_error_for_set6 = mean_squared_error(y_test, pred)

print("Mean Squared Error:", mean_squared_error(y_test, pred))
print("R2 Score:", r2_score(y_test, pred))

Mean Squared Error: 0.3721868055555556
R2 Score: 0.5815804129077029


In [ ]:
sample_index = 1
original_df_index = y_test.index[sample_index]

sample_essay = texts[original_df_index]
sample_actual_score = y_test.iloc[sample_index]
sample_predicted_score = pred[sample_index]

print(f"Original Essay:\n{sample_essay}\n")
print(f"Actual Score: {sample_actual_score}")
print(f"Predicted Score: {sample_predicted_score}")

Original Essay:
Many obstacles occured for the builers of the Empire State building and allowing dirigibles to dock there. One of these obstacles was the mooring mast itself. The winds on top of the Empire State Building were constantly shifting and if the dirigible was tied to the mooring mass it would swivel around the mooring mast. Another obstacle the builders experienced during this process was the already existing law not allowing airships to fly too low over urban areas. Due to the fact that the Empire State building was in New York, an urban area, the dirigable idea would have most likely never worked. The dirigable was unable to fulfill its destiny and the obstacles the builders faced were the main reasons.

Actual Score: 3
Predicted Score: 2.8


In [ ]:
sample_index = 25
original_df_index = y_test.index[sample_index]

sample_essay = texts[original_df_index]
sample_actual_score = y_test.iloc[sample_index]
sample_predicted_score = pred[sample_index]

print(f"Original Essay:\n{sample_essay}\n")
print(f"Actual Score: {sample_actual_score}")
print(f"Predicted Score: {sample_predicted_score}")

Original Essay:
In a desperate attempt to revolutionize the way people traveled, the idea of the mooring was not thought over as well as it should have been. The number of problems and obstacles mounted higher and higher with the idea of mooring airships to a new mast atop the Empire State Building. One of these main obstacles was, while an airship was tethered to the building it would add stress, "The stress of dirigibles load and the wind pressure would have to be transmitted all the way to the building's foundation". Another obstacle faced with the mooring mast was one of safety. Dirigibles not from the United States used hydrogen rather than helium, which is Highly flammable. Another obstacle to the construction was nature itself, with high, shifting winds due to violent air currents, a dirigible tethered to the mast would swivel around and around the Empire State Building, the only way to prevent this would be to tie the lead weights to the rear end, but massive dangling weights, 

# For Set 7

In [ ]:
df = data[data['essay_set'] == 7]
df = df[['essay', 'domain1_score']]

df = df.dropna()

df.head()

,essay,domain1_score
10684,Patience is when your waiting .I was patience ...,15
10685,"I am not a patience person, like I cant sit i...",13
10686,One day I was at basketball practice and I was...,15
10687,I going to write about a time when I went to t...,17
10688,It can be very hard for somebody to be patient...,13


In [ ]:
df.rename(columns={
    'essay': 'text',
    'domain1_score': 'score'
}, inplace=True)

In [ ]:
texts = df['text']
scores = df['score']

In [ ]:
embeddings = model.encode(
    texts.tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    embeddings,
    scores,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 1255
Testing samples: 314


In [ ]:
from sklearn.ensemble import RandomForestRegressor

regressor = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

regressor.fit(X_train, y_train)

RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42)

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

pred = regressor.predict(X_test)
mean_error_for_set7 = mean_squared_error(y_test, pred)

print("Mean Squared Error:", mean_squared_error(y_test, pred))
print("R2 Score:", r2_score(y_test, pred))

Mean Squared Error: 9.590237101910828
R2 Score: 0.533118142968028


In [ ]:
sample_index = 1
original_df_index = y_test.index[sample_index]

sample_essay = texts[original_df_index]
sample_actual_score = y_test.iloc[sample_index]
sample_predicted_score = pred[sample_index]

print(f"Original Essay:\n{sample_essay}\n")
print(f"Actual Score: {sample_actual_score}")
print(f"Predicted Score: {sample_predicted_score}")

Original Essay:
Once upon a time there was a young girl named @CAPS1 she lived with her dad and mom in a small town of @PERSON2. It was mom in a small town of @PERSON2. It was almost her birth and she was so exited for this birthday because she was turning twelve. Twelve was here luckey number ever sience she was little. Her mom said they could go into town and go to chef @PERSON1 and eat out she agreed so that morning she woke up and dad made her some pancakes with fresh strawberrys on them she went out to go milk there cows for her parents. @CAPS1 hated milking there cows because they are very stuburn so she went out there and miked the cows in only @NUM1 minetts. @CAPS1 went back inside and put the milk on the table. @CAPS1 was starting to wonder if she was going  to get a birthday present, @CAPS1 decided it would be best if she was patient when her mom came home from work (she works at a dress making store) she said, Are you ready to go eat, @CAPS1 jumped And yeld yes, yes. So t

In [ ]:
sample_index = 17
original_df_index = y_test.index[sample_index]

sample_essay = texts[original_df_index]
sample_actual_score = y_test.iloc[sample_index]
sample_predicted_score = pred[sample_index]

print(f"Original Essay:\n{sample_essay}\n")
print(f"Actual Score: {sample_actual_score}")
print(f"Predicted Score: {sample_predicted_score}")

Original Essay:
A time when I was patient. Two weeks ago @CAPS4 @CAPS2 said her @CAPS1 was coming. @CAPS4 grandma was happy so she bought her sister a bus ticket. Bus tickets cost a lot of money. It is @NUM1 dollar. That was a lot of money. She @CAPS11't miss this bus because I really want to see her. I could not wait. So she was suppose to be coming on @DATE1. She didn't come. I was sick so I called @CAPS4 @CAPS2 at her job. I said, "@CAPS2 why didn't @CAPS3 come." @CAPS4 @CAPS2 said," @CAPS5 she is coming you just have to be patient." @CAPS6, I said. Next day was @DATE2. "@CAPS2 is she here yet" I said. "@CAPS8, @CAPS5 be patient she will be here tomorrow." "@CAPS6, I understand."I gave up I thought that she was not coming. So @CAPS4 mom called @CAPS4 godmom. She said it was @CAPS6. So she told me she was going to @ORGANIZATION1. I love @ORGANIZATION1. I so good. I asked @CAPS4 @CAPS2. "@CAPS11 I have a @CAPS12. @ORGANIZATION2 cheese burger. I said @CAPS13 to @CAPS4 uncle and @CAPS4 

# For Set 8

In [ ]:
df = data[data['essay_set'] == 8]
df = df[['essay', 'domain1_score']]

df = df.dropna()

df.head()

,essay,domain1_score
12253,A long time ago when I was in third grade I h...,34
12254,Softball has to be one of the single most gre...,46
12255,"Some people like making people laugh, I love ...",40
12256,"""LAUGHTER"" @CAPS1 I hang out with my friends...",30
12257,Well ima tell a story about the time i got @CA...,26


In [ ]:
df.rename(columns={
    'essay': 'text',
    'domain1_score': 'score'
}, inplace=True)

In [ ]:
texts = df['text']
scores = df['score']

In [ ]:
embeddings = model.encode(
    texts.tolist(),
    show_progress_bar=True
)

Batches:   0%|          | 0/23 [00:00<?, ?it/s]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    embeddings,
    scores,
    test_size=0.2,
    random_state=42
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 578
Testing samples: 145


In [ ]:
from sklearn.ensemble import RandomForestRegressor

regressor = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

regressor.fit(X_train, y_train)

RandomForestRegressor(n_estimators=200, n_jobs=-1, random_state=42)

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

pred = regressor.predict(X_test)
mean_error_for_set8 = mean_squared_error(y_test, pred)

print("Mean Squared Error:", mean_squared_error(y_test, pred))
print("R2 Score:", r2_score(y_test, pred))

Mean Squared Error: 21.360693620689656
R2 Score: 0.23014796205667065


In [ ]:
sample_index = 8
original_df_index = y_test.index[sample_index]

sample_essay = texts[original_df_index]
sample_actual_score = y_test.iloc[sample_index]
sample_predicted_score = pred[sample_index]

print(f"Original Essay:\n{sample_essay}\n")
print(f"Actual Score: {sample_actual_score}")
print(f"Predicted Score: {sample_predicted_score}")

Original Essay:
                            Whats Laughter!How many friends do you really have? What relationships do you have with others? Well, I'm in high school and I usually don't have that many friends. But I have a few and one of my friends @CAPS4 friends with someone that I truly love. (@CAPS1 didn't know that) @CAPS2 year I was a freshmen and i went to the church with some of my friends and their was an @NUM1 grader that fell in the pond/water and I wasn't sure @CAPS20 @CAPS1 was @CAPS3 or not so I end up coming up to him and asked.Me; @CAPS4 you @CAPS3?, do you need some help?@CAPS1 said; @CAPS6, its @CAPS3 I just sprained my ankle.Me; @CAPS3, see you later then.Him. Later thanks for asking though.me; @CAPS9 sure, @CAPS6 problem! The next year when I'm a sophomore I didn't realized that @CAPS1 was going to be their as a freshmen! his name is @CAPS11 I was starting to hang out with him more because @CAPS1 was really funny and sweet.  We went to the @CAPS12 ball together and ou

In [ ]:
sample_index = 56
original_df_index = y_test.index[sample_index]

sample_essay = texts[original_df_index]
sample_actual_score = y_test.iloc[sample_index]
sample_predicted_score = pred[sample_index]

print(f"Original Essay:\n{sample_essay}\n")
print(f"Actual Score: {sample_actual_score}")
print(f"Predicted Score: {sample_predicted_score}")

Original Essay:
@DATE1 what a night!! Me and my friends we were going to a @CAPS1 party at our dance coaches work. we had to dress up as elves and play with kids. It was fun, but after we were done with the party that's when the real fun began! After we changed out of our costumes and back into our regular clothes we started taking pictures. @CAPS3 were funny ones,cute one, and weird ones,all kinds of pictures. Anyways my friends and i didn't want to go home just yet so we decided that we should go to some places to take more pictures and have more fun! First stop was a school play ground but we couldn't stay because it was late and we weren't allowed inside so we decided to go to the discovery park i was driving and one of my friends in the passenger seat was taking pictures that turned out really funny. so when we got to the park my friend that was taking the pictures said  @CAPS2 i drive around the parking lot?" I said sure. we switched spots and off we went. She was a pretty good d

# Final Result

In [ ]:
average_mse = (
    mean_error_for_set1 +
    mean_error_for_set2 +
    mean_error_for_set3 +
    mean_error_for_set4 +
    mean_error_for_set5 +
    mean_error_for_set6 +
    mean_error_for_set7 +
    mean_error_for_set8
) / 8

print("Average Mean Squared Error:", average_mse)

Average Mean Squared Error: 4.295766024728887
